In [13]:
import pathlib
import xarray as xr
import matplotlib.pyplot as plt
import global3d_track as g3d

In [9]:
# make masked data

In [11]:
# load mask
fdir = '/work/bb1153/b382635/data/final_tracks/dcc_dev/anatomy_example_large/20210701/20210701T0000_20210702T0000_system_tracks.nc'
mask = xr.open_dataset(fdir)
mask

<xarray.Dataset> Size: 930MB
Dimensions:     (time: 95, level_full: 51, lat: 100, lon: 120)
Coordinates:
  * time        (time) datetime64[ns] 760B 2021-07-01T00:15:00 ... 2021-07-01...
  * level_full  (level_full) int32 204B 40 41 42 43 44 45 ... 85 86 87 88 89 90
  * lat         (lat) float64 800B 7.0 7.1 7.2 7.3 7.4 ... 16.6 16.7 16.8 16.9
  * lon         (lon) float64 960B -80.0 -79.9 -79.8 -79.7 ... -68.3 -68.2 -68.1
Data variables:
    system      (time, level_full, lat, lon) int64 465MB ...
    u_tracks    (time, level_full, lat, lon) int64 465MB ...

In [17]:
# load data
data = g3d.utils.data_tools.load_corresponding_data(mask, variables=None)

INFO:root:Region to regrid: (-80.0, -68.00000000000068, 7.0, 16.999999999999964)


In [18]:
data

<xarray.Dataset> Size: 5GB
Dimensions:     (time: 95, level_full: 90, lat: 100, lon: 120, level_half: 91)
Coordinates:
  * level_full  (level_full) int32 360B 1 2 3 4 5 6 7 8 ... 84 85 86 87 88 89 90
  * level_half  (level_half) int32 364B 1 2 3 4 5 6 7 8 ... 85 86 87 88 89 90 91
  * time        (time) datetime64[ns] 760B 2021-07-01T00:15:00 ... 2021-07-01...
    zg          (level_full, lat, lon) float32 4MB ...
    zghalf      (level_half, lat, lon) float32 4MB ...
  * lat         (lat) float64 800B 7.0 7.1 7.2 7.3 7.4 ... 16.6 16.7 16.8 16.9
  * lon         (lon) float64 960B -80.0 -79.9 -79.8 -79.7 ... -68.3 -68.2 -68.1
Data variables: (12/15)
    cli         (time, level_full, lat, lon) float32 410MB ...
    clw         (time, level_full, lat, lon) float32 410MB ...
    dzghalf     (level_full, lat, lon) float32 4MB ...
    hus         (time, level_full, lat, lon) float32 410MB ...
    pfull       (time, level_full, lat, lon) float32 410MB ...
    pr          (time, lat, lon) float32 5MB ...
    ...          ...
    rlut        (time, lat, lon) float32 5MB ...
    ta          (time, level_full, lat, lon) float32 410MB ...
    ts          (time, lat, lon) float32 5MB ...
    ua          (time, level_full, lat, lon) float32 410MB ...
    va          (time, level_full, lat, lon) float32 410MB ...
    wa_phy      (time, level_half, lat, lon) float32 415MB ...

In [ ]:
# save data 
dpath = '/work/bb1153/b382635/data/final_tracks/dcc_dev/anatomy_example_large/20210701/20210701T0000_20210702T0000_model_fields.nc'
data.to_netcdf(dpath)


In [ ]:
# load data

dpath = '/work/bb1153/b382635/data/final_tracks/dcc_dev/anatomy_example_large/20210701/20210701T0000_20210702T0000_model_fields.nc'
data = xr.open_dataset(dpath)
data

In [ ]:
# view tracked DCC labels (0 is the background)
np.unique(mask.system)

In [ ]:
# view one DCC
sidx = 1
dcc = data.where(mask.system==1).dropna('lat', how='all').dropna('lon', how='all')

In [ ]:
# max updraft
dcc.wa_phy.max(('level_full','time')).plot()

In [ ]:
# updraft vs condesate
w = dcc.wa_phy.max(('level_full','lat','lon')) # m s-1
tot_q = xr.concat([dcc[q] for q in quantity], dim='q').sum('q', skipna=True) # kg kg-1
q = tot_q.mean(('lat','lon','level_full'))

In [ ]:
fig, ax = plt.subplots()

w.plot(ax=ax, c='r', label='max updraft')
q.plot(ax=ax.twinx(), c='b', label='mean specific condensate')
ax.legend()